In [1]:
# ==========================================
# 1. LIBRERÍAS ESTÁNDAR Y GENERALES
# ==========================================
import os
import re
import random
import requests as req

# ==========================================
# 2. PROCESAMIENTO DE DATOS Y VISUALIZACIÓN
# ==========================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 3. SCIKIT-LEARN: PREPROCESAMIENTO Y SELECCIÓN
# ==========================================
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA

# ==========================================
# 4. SCIKIT-LEARN: MODELOS (Clasificación, Regresión y Clustering)
# ==========================================
# Clasificación
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Regresión
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Clustering
from sklearn.cluster import KMeans

# ==========================================
# 5. SCIKIT-LEARN: MÉTRICAS DE EVALUACIÓN
# ==========================================
# Métricas generales y de Clasificación
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    recall_score, 
    precision_score
)

# Métricas de Regresión
from sklearn.metrics import (
    mean_squared_error, 
    mean_absolute_error, 
    r2_score
)

from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# ==========================================
# 1. CONFIGURACIÓN Y CARGA
# ==========================================
file_path_nat = '12612-0100_en_natalidad.csv'
# Quitamos el skipfooter por ahora para asegurar que no cortamos a Turingia
df_natalidad = pd.read_csv(file_path_nat, sep=';', skiprows=4, engine='python')

# ==========================================
# 2. LIMPIEZA DE CABECERAS Y RELLENO (ffill)
# ==========================================
df_natalidad.rename(columns={df_natalidad.columns[0]: 'Estado_DE', df_natalidad.columns[1]: 'Sexo'}, inplace=True)

# Limpiamos espacios y basura en las columnas clave
df_natalidad['Estado_DE'] = df_natalidad['Estado_DE'].astype(str).str.strip()
df_natalidad['Sexo'] = df_natalidad['Sexo'].astype(str).str.strip()

# Rellenamos los nombres de los estados
df_natalidad['Estado_DE'] = df_natalidad['Estado_DE'].replace('', np.nan).replace('nan', np.nan).ffill()

# FILTRO DE SEGURIDAD: Antes de filtrar por Total, veamos si Turingia existe
# (Si no aparece aquí, es que el CSV no la tiene o el skiprows la borró)
print("¿Está la palabra Thüringen o similar en el archivo original?:")
print(df_natalidad[df_natalidad['Estado_DE'].str.contains('Th', na=False)]['Estado_DE'].unique())

# Ahora sí, filtramos
df_natalidad = df_natalidad[df_natalidad['Sexo'] == 'Total'].copy()

# ==========================================
# 3. SELECCIÓN DE COLUMNAS LIMPIAS
# ==========================================
cols_validas = ['Estado_DE'] + [col for col in df_natalidad.columns if col.isdigit()]
df_natalidad = df_natalidad[cols_validas]

# ==========================================
# 4. TRADUCCIÓN Y CATEGORIZACIÓN
# ==========================================
mapa_regiones = {
    'Baden-WÃ¼rttemberg': 'Baden-Wurtemberg', 'Baden-Württemberg': 'Baden-Wurtemberg',
    'Bayern': 'Baviera', 'Berlin': 'Berlin', 'Brandenburg': 'Brandeburgo', 
    'Bremen': 'Bremen', 'Hamburg': 'Hamburgo', 'Hessen': 'Hesse', 
    'Mecklenburg-Vorpommern': 'Mecklemburgo-Pomerania', 'Niedersachsen': 'Baja Sajonia', 
    'Nordrhein-Westfalen': 'Renania del Norte-Westfalia', 'Rheinland-Pfalz': 'Renania-Palatinado', 
    'Saarland': 'Sarre', 'Sachsen': 'Sajonia', 'Sachsen-Anhalt': 'Sajonia-Anhalt', 
    'Schleswig-Holstein': 'Schleswig-Holstein', 'Thüringen': 'Turingia', 'ThÃ¼ringen': 'Turingia'
}

df_natalidad['Estado'] = df_natalidad['Estado_DE'].map(mapa_regiones)

# RESCATE FINAL: Si sigue sin nombre, lo forzamos por posición si es necesario 
# o por coincidencia parcial de texto.
df_natalidad.loc[df_natalidad['Estado'].isna() & df_natalidad['Estado_DE'].str.contains('Th|TÃ|ThÃ', na=False), 'Estado'] = 'Turingia'

# Asignar Región
estados_este = ['Brandeburgo', 'Mecklemburgo-Pomerania', 'Sajonia', 'Sajonia-Anhalt', 'Turingia']
df_natalidad['Region_Tipo'] = df_natalidad['Estado'].apply(
    lambda x: 'Berlin' if x == 'Berlin' else ('Este' if x in estados_este else 'Oeste')
)

# Eliminamos filas que no tengan Estado (como el total nacional de Alemania)
df_natalidad = df_natalidad.dropna(subset=['Estado'])

# ==========================================
# 5. VERIFICACIÓN FINAL
# ==========================================
print(f"\nTotal de estados: {len(df_natalidad)}")
print("Lista final:", df_natalidad['Estado'].unique())

¿Está la palabra Thüringen o similar en el archivo original?:
['Thüringen']

Total de estados: 16
Lista final: ['Baden-Wurtemberg' 'Baviera' 'Berlin' 'Brandeburgo' 'Bremen' 'Hamburgo'
 'Hesse' 'Mecklemburgo-Pomerania' 'Baja Sajonia'
 'Renania del Norte-Westfalia' 'Renania-Palatinado' 'Sarre' 'Sajonia'
 'Sajonia-Anhalt' 'Schleswig-Holstein' 'Turingia']


In [3]:
df_natalidad.head(20)

,Estado_DE,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2017,2018,2019,2020,2021,2022,2023,2024,Estado,Region_Tipo
2,Baden-Württemberg,118579.0,117528.0,117559.0,117982.0,113398.0,112459.0,114657.0,116419.0,111056.0,...,107375.0,108919.0,108985.0,108024.0,113534.0,104549.0,98419.0,97507.0,Baden-Wurtemberg,Oeste
5,Bayern,136122.0,134400.0,133946.0,133897.0,127828.0,125995.0,129376.0,130517.0,126529.0,...,126187.0,127616.0,128227.0,128764.0,134321.0,124897.0,116505.0,114365.0,Baviera,Oeste
8,Berlin,37596.0,30562.0,29667.0,28724.0,28503.0,28648.0,29905.0,30369.0,29612.0,...,40160.0,40203.0,39503.0,38693.0,39168.0,35729.0,34120.0,33749.0,Berlin,Berlin
11,Brandenburg,29238.0,17215.0,13469.0,12238.0,12443.0,13494.0,15140.0,16370.0,17146.0,...,20337.0,19881.0,19329.0,18998.0,19029.0,17439.0,15885.0,15154.0,Brandeburgo,Este
14,Bremen,6895.0,6789.0,6757.0,6656.0,6288.0,6429.0,6623.0,6644.0,6360.0,...,7000.0,7163.0,7149.0,6968.0,6971.0,6720.0,6615.0,6251.0,Bremen,Oeste
17,Hamburg,16693.0,16503.0,16497.0,16257.0,16201.0,15872.0,16594.0,16970.0,16235.0,...,21133.0,21126.0,20940.0,20431.0,21018.0,19054.0,18264.0,17553.0,Hamburgo,Oeste
20,Hessen,62026.0,61324.0,61146.0,61610.0,60565.0,59858.0,62391.0,63124.0,60567.0,...,60988.0,61012.0,60062.0,59389.0,61546.0,57360.0,53685.0,53089.0,Hesse,Oeste
23,Mecklenburg-Vorpommern,23503.0,13635.0,10875.0,9432.0,8934.0,9878.0,11088.0,12046.0,12246.0,...,13081.0,13032.0,12630.0,12061.0,11845.0,10820.0,9671.0,9157.0,Mecklemburgo-Pomerania,Este
26,Niedersachsen,82452.0,83122.0,83669.0,84579.0,81520.0,80994.0,83655.0,85907.0,82207.0,...,73020.0,73652.0,73286.0,74119.0,76441.0,71289.0,67162.0,65646.0,Baja Sajonia,Oeste
29,Nordrhein-Westfalen,199294.0,198436.0,196899.0,194156.0,186079.0,182393.0,188493.0,190386.0,182287.0,...,171979.0,173150.0,170391.0,170038.0,175386.0,164496.0,155515.0,152688.0,Renania del Norte-Westfalia,Oeste


In [4]:
# ==========================================
# CREACIÓN DE MAESTRO DE ESTADOS (1-16)
# ==========================================

# 1. Obtenemos los nombres únicos de los estados que ya limpiamos
nombres_estados = sorted(df_natalidad['Estado'].unique())

# 2. Creamos un DataFrame maestro con su numeración
df_estados_id = pd.DataFrame({
    'ID_Estado': range(1, 17),  # Genera números del 1 al 16
    'Estado': nombres_estados
})

# 3. Unimos este ID a nuestra tabla de natalidad actual
df_natalidad = pd.merge(df_estados_id, df_natalidad, on='Estado')

# 4. Verificamos la numeración
print("✅ Estados numerados del 1 al 16:")
print(df_estados_id)

print("\n--- Vista previa de df_natalidad con ID ---")
print(df_natalidad[['ID_Estado', 'Estado', 'Region_Tipo', '1990', '2023']].head())

✅ Estados numerados del 1 al 16:
    ID_Estado                       Estado
0           1             Baden-Wurtemberg
1           2                 Baja Sajonia
2           3                      Baviera
3           4                       Berlin
4           5                  Brandeburgo
5           6                       Bremen
6           7                     Hamburgo
7           8                        Hesse
8           9       Mecklemburgo-Pomerania
9          10  Renania del Norte-Westfalia
10         11           Renania-Palatinado
11         12                      Sajonia
12         13               Sajonia-Anhalt
13         14                        Sarre
14         15           Schleswig-Holstein
15         16                     Turingia

--- Vista previa de df_natalidad con ID ---
   ID_Estado            Estado Region_Tipo      1990      2023
0          1  Baden-Wurtemberg       Oeste  118579.0   98419.0
1          2      Baja Sajonia       Oeste   82452.0   67162.0
2 

In [5]:
df_natalidad.head(20)

,ID_Estado,Estado,Estado_DE,1990,1991,1992,1993,1994,1995,1996,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Region_Tipo
0,1,Baden-Wurtemberg,Baden-Württemberg,118579.0,117528.0,117559.0,117982.0,113398.0,112459.0,114657.0,...,107487.0,107375.0,108919.0,108985.0,108024.0,113534.0,104549.0,98419.0,97507.0,Oeste
1,2,Baja Sajonia,Niedersachsen,82452.0,83122.0,83669.0,84579.0,81520.0,80994.0,83655.0,...,75215.0,73020.0,73652.0,73286.0,74119.0,76441.0,71289.0,67162.0,65646.0,Oeste
2,3,Baviera,Bayern,136122.0,134400.0,133946.0,133897.0,127828.0,125995.0,129376.0,...,125686.0,126187.0,127616.0,128227.0,128764.0,134321.0,124897.0,116505.0,114365.0,Oeste
3,4,Berlin,Berlin,37596.0,30562.0,29667.0,28724.0,28503.0,28648.0,29905.0,...,41086.0,40160.0,40203.0,39503.0,38693.0,39168.0,35729.0,34120.0,33749.0,Berlin
4,5,Brandeburgo,Brandenburg,29238.0,17215.0,13469.0,12238.0,12443.0,13494.0,15140.0,...,20934.0,20337.0,19881.0,19329.0,18998.0,19029.0,17439.0,15885.0,15154.0,Este
5,6,Bremen,Bremen,6895.0,6789.0,6757.0,6656.0,6288.0,6429.0,6623.0,...,7136.0,7000.0,7163.0,7149.0,6968.0,6971.0,6720.0,6615.0,6251.0,Oeste
6,7,Hamburgo,Hamburg,16693.0,16503.0,16497.0,16257.0,16201.0,15872.0,16594.0,...,21480.0,21133.0,21126.0,20940.0,20431.0,21018.0,19054.0,18264.0,17553.0,Oeste
7,8,Hesse,Hessen,62026.0,61324.0,61146.0,61610.0,60565.0,59858.0,62391.0,...,60731.0,60988.0,61012.0,60062.0,59389.0,61546.0,57360.0,53685.0,53089.0,Oeste
8,9,Mecklemburgo-Pomerania,Mecklenburg-Vorpommern,23503.0,13635.0,10875.0,9432.0,8934.0,9878.0,11088.0,...,13442.0,13081.0,13032.0,12630.0,12061.0,11845.0,10820.0,9671.0,9157.0,Este
9,10,Renania del Norte-Westfalia,Nordrhein-Westfalen,199294.0,198436.0,196899.0,194156.0,186079.0,182393.0,188493.0,...,173274.0,171979.0,173150.0,170391.0,170038.0,175386.0,164496.0,155515.0,152688.0,Oeste


In [6]:
# ==========================================
# 0️⃣ Ver cómo están los nombres de columnas
# ==========================================
print("Columnas originales:")
print(df_natalidad.columns.tolist())

# ==========================================
# 1️⃣ Limpiar nombres de columnas
# ==========================================
df_natalidad.columns = df_natalidad.columns.str.strip()       # quitar espacios al inicio/final
df_natalidad.columns = df_natalidad.columns.str.replace('#', '')  # quitar #
df_natalidad.columns = df_natalidad.columns.str.replace(' ', '')  # quitar espacios intermedios si hay

print("\nColumnas limpias:")
print(df_natalidad.columns.tolist())

# ==========================================
# 2️⃣ Seleccionar solo columnas de años
# ==========================================
cols_años = [col for col in df_natalidad.columns if col.isdigit()]
print("\nColumnas de años detectadas:")
print(cols_años)

# ==========================================
# 3️⃣ Limpiar valores: quitar separadores de miles y convertir a float
# ==========================================
for col in cols_años:
    df_natalidad[col] = (
        df_natalidad[col]
        .astype(str)
        .str.replace(' ', '')      # quita espacio de miles
        .str.replace('.', '')      # quita punto de miles si aparece
        .replace('-', np.nan)      # reemplaza '-' por NaN
        .astype(float)
    )

# ==========================================
# 4️⃣ Renombrar columnas para merge futuro
# ==========================================
df_natalidad.rename(
    columns={col: f"{col}_natalidad" for col in cols_años},
    inplace=True
)

# ==========================================
# 5️⃣ Verificación final
# ==========================================
print("\nPrimeras filas después de limpiar y renombrar:")
print(df_natalidad.head(5))

print("\nTipos de datos:")
print(df_natalidad.dtypes)


Columnas originales:
['ID_Estado', 'Estado', 'Estado_DE', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', 'Region_Tipo']

Columnas limpias:
['ID_Estado', 'Estado', 'Estado_DE', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', 'Region_Tipo']

Columnas de años detectadas:
['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '

In [7]:
# ==========================================
# Eliminar columna 1990_natalidad
# ==========================================
if '1990_natalidad' in df_natalidad.columns:
    df_natalidad.drop(columns=['1990_natalidad'], inplace=True)

# Verificación
print("Columnas después de eliminar 1990_natalidad:")
print(df_natalidad.columns.tolist())


Columnas después de eliminar 1990_natalidad:
['ID_Estado', 'Estado', 'Estado_DE', '1991_natalidad', '1992_natalidad', '1993_natalidad', '1994_natalidad', '1995_natalidad', '1996_natalidad', '1997_natalidad', '1998_natalidad', '1999_natalidad', '2000_natalidad', '2001_natalidad', '2002_natalidad', '2003_natalidad', '2004_natalidad', '2005_natalidad', '2006_natalidad', '2007_natalidad', '2008_natalidad', '2009_natalidad', '2010_natalidad', '2011_natalidad', '2012_natalidad', '2013_natalidad', '2014_natalidad', '2015_natalidad', '2016_natalidad', '2017_natalidad', '2018_natalidad', '2019_natalidad', '2020_natalidad', '2021_natalidad', '2022_natalidad', '2023_natalidad', '2024_natalidad', 'Region_Tipo']


In [8]:
# PIB

# ==========================================
# 0️⃣ Maestro de estados (ID fijo)
# ==========================================
nombres_oficiales = [
    'Baden-Wurtemberg', 'Baviera', 'Berlin', 'Brandeburgo', 'Bremen', 'Hamburgo',
    'Hesse', 'Mecklemburgo-Pomerania', 'Baja Sajonia', 'Renania del Norte-Westfalia',
    'Renania-Palatinado', 'Sarre', 'Sajonia', 'Sajonia-Anhalt', 'Schleswig-Holstein', 'Turingia'
]

df_estados_id = pd.DataFrame({
    'ID_Estado': range(1, 17),
    'Estado': nombres_oficiales
})

# ==========================================
# 1️⃣ Carga y limpieza inicial del PIB
# ==========================================
file_path_pib = '82111-0010_en_pibregion.csv'
df_pib_raw = pd.read_csv(file_path_pib, sep=';', skiprows=6, engine='python')

df_pib_raw = df_pib_raw[
    df_pib_raw.iloc[:, 0].astype(str).str.isnumeric()
].copy()

# ==========================================
# 2️⃣ Selección de columnas (Año + estados)
# ==========================================
indices_columnas = [0] + list(range(1, 33, 2))
df_pib_raw = df_pib_raw.iloc[:, indices_columnas]

df_pib_raw.columns = ['Año'] + nombres_oficiales

# ==========================================
# 3️⃣ Transponer
# ==========================================
df_pib_raw['Año'] = df_pib_raw['Año'].astype(int).astype(str)

df_pib = (
    df_pib_raw
    .set_index('Año')
    .transpose()
    .reset_index()
    .rename(columns={'index': 'Estado'})
)

# ==========================================
# 4️⃣ Merge con maestro + región
# ==========================================
df_pib = pd.merge(df_estados_id, df_pib, on='Estado')

estados_este = ['Brandeburgo', 'Mecklemburgo-Pomerania', 'Sajonia', 'Sajonia-Anhalt', 'Turingia']

df_pib.insert(
    2,
    'Region_Tipo',
    df_pib['Estado'].apply(
        lambda x: 'Berlin' if x == 'Berlin'
        else ('Este' if x in estados_este else 'Oeste')
    )
)

# ==========================================
# 5️⃣ Ordenar por ID_Estado
# ==========================================
df_pib = df_pib.sort_values('ID_Estado').reset_index(drop=True)

# ==========================================
# 6️⃣ Convertir años a enteros
# ==========================================
columnas_años = [col for col in df_pib.columns if col.isdigit()]

for col in columnas_años:
    df_pib[col] = (
        pd.to_numeric(df_pib[col], errors='coerce')
        .fillna(0)
        .astype(int)
    )

# ==========================================
# 7️⃣ Renombrar columnas → _pib
# ==========================================
df_pib.rename(
    columns={col: f"{col}_pib" for col in columnas_años},
    inplace=True
)

# ==========================================
# 8️⃣ Verificación final
# ==========================================
print("✅ Dataset de PIB listo (Formato Ancho)")
print(f"Filas: {len(df_pib)} (Deberían ser 16)")
print(df_pib.head())

✅ Dataset de PIB listo (Formato Ancho)
Filas: 16 (Deberían ser 16)
   ID_Estado            Estado Region_Tipo  1991_pib  1992_pib  1993_pib  \
0          1  Baden-Wurtemberg       Oeste    243387    256691    254219   
1          2           Baviera       Oeste    264017    284019    288542   
2          3            Berlin      Berlin     68403     75606     81066   
3          4       Brandeburgo        Este     19850     24767     30101   
4          5            Bremen       Oeste     18991     19563     19470   

   1994_pib  1995_pib  1996_pib  1997_pib  ...  2015_pib  2016_pib  2017_pib  \
0    263619    272298    277785    283800  ...    468613    483231    506847   
1    298865    306896    312550    319670  ...    564087    586881    614257   
2     83845     86026     85339     83943  ...    129267    136366    144070   
3     34711     38091     39971     40688  ...     66755     68644     72111   
4     19955     20298     20352     21122  ...     31081     31910     32788

In [9]:
# Crear maestro de estados en el mismo orden que natalidad
nombres_estados = sorted(df_pib['Estado'].unique())

df_estados_id = pd.DataFrame({
    'ID_Estado': range(1, 17),
    'Estado': nombres_estados
})

# Quitar el ID_Estado actual
df_pib = df_pib.drop(columns='ID_Estado')

# Volver a unir para que tome el ID correcto
df_pib = pd.merge(df_estados_id, df_pib, on='Estado')

# Ordenar por el nuevo ID
df_pib = df_pib.sort_values('ID_Estado').reset_index(drop=True)

In [10]:
df_pib.columns = df_pib.columns.str.strip().str.replace('#', '')

In [11]:
df_pib.head(20)

,ID_Estado,Estado,Region_Tipo,1991_pib,1992_pib,1993_pib,1994_pib,1995_pib,1996_pib,1997_pib,...,2015_pib,2016_pib,2017_pib,2018_pib,2019_pib,2020_pib,2021_pib,2022_pib,2023_pib,2024_pib
0,1,Baden-Wurtemberg,Oeste,243387,256691,254219,263619,272298,277785,283800,...,468613,483231,506847,526436,536086,516888,555660,595351,631540,650225
1,2,Baja Sajonia,Oeste,145213,154134,157046,163902,165557,166744,169683,...,265503,285288,292612,303764,314460,306655,321135,343641,369147,381267
2,3,Baviera,Oeste,264017,284019,288542,298865,306896,312550,319670,...,564087,586881,614257,629022,651194,634843,675004,723639,773647,791603
3,4,Berlin,Berlin,68403,75606,81066,83845,86026,85339,83943,...,129267,136366,144070,152030,159591,159011,171190,184512,197924,207058
4,5,Brandeburgo,Este,19850,24767,30101,34711,38091,39971,40688,...,66755,68644,72111,73970,77181,76206,81192,89534,96433,97540
5,6,Bremen,Oeste,18991,19563,19470,19955,20298,20352,21122,...,31081,31910,32788,33389,33433,32442,35483,38820,40338,41357
6,7,Hamburgo,Oeste,61137,63737,65919,67741,69210,70759,73470,...,112880,115167,121447,124313,129913,124166,137640,155671,153737,161856
7,8,Hesse,Oeste,150385,159369,161858,166412,170701,175094,178574,...,268527,279760,288020,294358,302764,293847,313308,332353,354492,368298
8,9,Mecklemburgo-Pomerania,Este,14436,17800,21337,24713,26937,27907,28336,...,40823,41786,45069,45313,48230,47292,50164,55661,59168,61245
9,10,Renania del Norte-Westfalia,Oeste,381575,402569,404924,417105,430715,432610,442335,...,651068,666208,693415,716741,731488,717488,754396,806852,851036,871867


In [12]:
# =========================================================
# PROCESAMIENTO RIGUROSO DEL PARO (Formato Ancho)
# =========================================================

# 1. Carga usando índices de columna reales
file_path_paro = '13211-0007_en_paro.csv'
# Saltamos las primeras 5 filas de basura y leemos sin cabeceras
df_paro_raw = pd.read_csv(file_path_paro, sep=';', skiprows=5, engine='python', header=None)

# Según el archivo:
# Columna 0: Estados / Años
# Columna 1: Registered unemployed (number)
# Columna 5: Rate of registered unemployed (percent)
df_paro_raw = df_paro_raw[[0, 1, 5]].copy()

# 2. Identificación de bloques por Estado
# Convertimos la columna 0 a string para detectar los años
df_paro_raw[0] = df_paro_raw[0].astype(str).str.strip()
es_año = df_paro_raw[0].str.isnumeric()

# Creamos la columna de Estado basada en las filas que NO son años
df_paro_raw['Estado_Origen'] = df_paro_raw[0].where(~es_año).ffill()

# Filtramos para quedarnos solo con las filas de datos (las que tienen año en la col 0)
df_paro_clean = df_paro_raw[es_año].copy()

# 3. Traducción de los nombres de los estados (para que coincidan con tus IDs)
mapa_paro = {
    'Baden-Württemberg': 'Baden-Wurtemberg', 'Bayern': 'Baviera', 'Berlin': 'Berlin',
    'Brandenburg': 'Brandeburgo', 'Bremen': 'Bremen', 'Hamburg': 'Hamburgo',
    'Hessen': 'Hesse', 'Mecklenburg-Vorpommern': 'Mecklemburgo-Pomerania',
    'Niedersachsen': 'Baja Sajonia', 'Nordrhein-Westfalen': 'Renania del Norte-Westfalia',
    'Rheinland-Pfalz': 'Renania-Palatinado', 'Saarland': 'Sarre', 'Sachsen': 'Sajonia',
    'Sachsen-Anhalt': 'Sajonia-Anhalt', 'Schleswig-Holstein': 'Schleswig-Holstein',
    'Thüringen': 'Turingia'
}
df_paro_clean['Estado'] = df_paro_clean['Estado_Origen'].map(mapa_paro)

# 4. Limpieza de datos (Eliminar 'e', guiones y convertir a número)
for col_idx in [1, 5]:
    df_paro_clean[col_idx] = pd.to_numeric(
        df_paro_clean[col_idx].astype(str).str.replace('-', '0'), 
        errors='coerce'
    ).fillna(0)

# =========================================================
# 5. CREACIÓN DE LAS DOS TABLAS MAESTRAS (16 FILAS CADA UNA)
# =========================================================

# --- TABLA A: Registered unemployed (number) ---
df_paro_numero = df_paro_clean.pivot(index='Estado', columns=0, values=1).reset_index()
df_paro_numero = pd.merge(df_estados_id, df_paro_numero, on='Estado')

# --- TABLA B: Rate of registered unemployed (percent) ---
df_paro_tasa = df_paro_clean.pivot(index='Estado', columns=0, values=5).reset_index()
df_paro_tasa = pd.merge(df_estados_id, df_paro_tasa, on='Estado')

# 6. Inserción de Region_Tipo y limpieza de tipos de datos
for df in [df_paro_numero, df_paro_tasa]:
    df.insert(2, 'Region_Tipo', df['Estado'].apply(
        lambda x: 'Berlin' if x == 'Berlin' else ('Este' if x in estados_este else 'Oeste')
    ))
    
    # Aseguramos que los años sean columnas de tipo entero/float según corresponda
    años_detectados = [c for c in df.columns if str(c).isdigit()]
    for año in años_detectados:
        if df is df_paro_numero:
            df[año] = df[año].astype(int)
        else:
            df[año] = df[año].astype(float)

# ==========================================
# RESULTADO FINAL
# ==========================================
print("✅ Tabla de Número de Parados (16 filas):")
print(df_paro_numero.head(2))

print("\n✅ Tabla de Tasa de Paro % (16 filas):")
print(df_paro_tasa.head(2))

✅ Tabla de Número de Parados (16 filas):
   ID_Estado            Estado Region_Tipo    1991    1992    1993    1994  \
0          1  Baden-Wurtemberg       Oeste  159318  191970  281496  333416   
1          2      Baja Sajonia       Oeste  244283  249792  306848  340822   

     1995    1996    1997  ...    2016    2017    2018    2019    2020  \
0  328298  353919  382008  ...  226421  212837  195128  196950  259940   
1  346948  386244  413832  ...  252574  244260  227834  218123  251377   

     2021    2022    2023    2024    2025  
0  247774  223119  245466  269990  293713  
1  243021  230553  251873  263090  273342  

[2 rows x 38 columns]

✅ Tabla de Tasa de Paro % (16 filas):
   ID_Estado            Estado Region_Tipo  1991  1992  1993  1994  1995  \
0          1  Baden-Wurtemberg       Oeste   3.7   4.4   6.3   7.5   7.4   
1          2      Baja Sajonia       Oeste   8.1   8.1   9.7  10.7  10.9   

   1996  1997  ...  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025

In [13]:
df_paro_numero.head(16)

,ID_Estado,Estado,Region_Tipo,1991,1992,1993,1994,1995,1996,1997,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,1,Baden-Wurtemberg,Oeste,159318,191970,281496,333416,328298,353919,382008,...,226421,212837,195128,196950,259940,247774,223119,245466,269990,293713
1,2,Baja Sajonia,Oeste,244283,249792,306848,340822,346948,386244,413832,...,252574,244260,227834,218123,251377,243021,230553,251873,263090,273342
2,3,Baviera,Oeste,214780,243224,322667,360862,355881,401991,442283,...,250623,231353,214017,211965,275075,262186,235851,257096,285234,315289
3,4,Berlin,Berlin,179953,207058,203924,210130,213383,235999,265665,...,181018,168991,156230,152565,192644,198401,179327,187930,203242,218315
4,5,Brandeburgo,Este,141172,182342,180418,178842,164608,187051,218148,...,105555,92648,83669,76888,82491,78463,74242,78996,82795,86413
5,6,Bremen,Oeste,31629,31532,36638,40155,40343,44374,46951,...,36393,35687,34904,35702,40822,39292,37214,39050,41116,43293
6,7,Hamburgo,Oeste,63013,57441,62929,71204,77773,83942,92520,...,70666,69248,65589,64774,80677,80395,73800,80806,88471,93783
7,8,Hesse,Oeste,123264,136825,177061,208675,213222,234083,260796,...,172826,166287,154337,149812,184955,178086,164492,181344,194912,205944
8,9,Mecklemburgo-Pomerania,Este,128303,163163,151996,143600,132850,147786,168364,...,80389,70982,64993,58485,63850,62410,59571,63191,64944,65605
9,10,Renania del Norte-Westfalia,Oeste,561331,580473,703198,784510,778946,826959,884479,...,725653,701219,650768,635486,733740,718220,668502,710175,749712,783416


In [14]:
# =========================================================
# LIMPIEZA FINAL DEL PARO (1991-2024 + Renombrado)
# =========================================================

# 1. Filtramos los años: nos quedamos solo con el rango 1991-2024
# Buscamos todas las columnas que sean números
años_en_tabla = [c for c in df_paro_numero.columns if str(c).isdigit()]
# Filtramos para que solo entren los que están entre 1991 y 2024
años_validos = [c for c in años_en_tabla if 1991 <= int(c) <= 2024]

# 2. Reconstruimos la tabla solo con las columnas deseadas
columnas_finales = ['ID_Estado', 'Estado', 'Region_Tipo'] + años_validos
df_paro_numero = df_paro_numero[columnas_finales].copy()

# 3. RENOMBRADO DE COLUMNAS (Añadiendo _paro)
# Creamos un diccionario que diga: {'1991': '1991_paro', '1992': '1992_paro'...}
nuevos_nombres = {año: f"{año}_paro" for año in años_validos}
df_paro_numero.rename(columns=nuevos_nombres, inplace=True)

# 4. Verificación de seguridad
print("✅ Tabla de Paro (Número) finalizada:")
print(f"Columnas actuales: {df_paro_numero.columns[:5].tolist()} ... {df_paro_numero.columns[-1]}")
print(df_paro_numero.head(16))

# --- OPCIONAL: Repetir para la Tasa si la necesitas ---
df_paro_tasa = df_paro_tasa[['ID_Estado', 'Estado', 'Region_Tipo'] + años_validos].copy()
nuevos_nombres_tasa = {año: f"{año}_tasa_paro" for año in años_validos}
df_paro_tasa.rename(columns=nuevos_nombres_tasa, inplace=True)

✅ Tabla de Paro (Número) finalizada:
Columnas actuales: ['ID_Estado', 'Estado', 'Region_Tipo', '1991_paro', '1992_paro'] ... 2024_paro
    ID_Estado                       Estado Region_Tipo  1991_paro  1992_paro  \
0           1             Baden-Wurtemberg       Oeste     159318     191970   
1           2                 Baja Sajonia       Oeste     244283     249792   
2           3                      Baviera       Oeste     214780     243224   
3           4                       Berlin      Berlin     179953     207058   
4           5                  Brandeburgo        Este     141172     182342   
5           6                       Bremen       Oeste      31629      31532   
6           7                     Hamburgo       Oeste      63013      57441   
7           8                        Hesse       Oeste     123264     136825   
8           9       Mecklemburgo-Pomerania        Este     128303     163163   
9          10  Renania del Norte-Westfalia       Oeste     561331

In [15]:

# 1. CARGA
file_path_voto = 'btw_ab49_datenbank_ergebnisse (1)_voto.csv'
df_voto_raw = pd.read_csv(file_path_voto, sep=';', skiprows=8, header=None, dtype=str, engine='python')

# 2. MAPEO DE COLUMNAS (Índice Python = Excel - 1)
col_map_voto = {
    'Baden-Wurtemberg': 10, 'Baviera': 14, 'Berlin': 18, 'Brandeburgo': 22,
    'Bremen': 26, 'Hamburgo': 30, 'Hesse': 34, 'Mecklemburgo-Pomerania': 38,
    'Baja Sajonia': 42, 'Renania del Norte-Westfalia': 46, 'Renania-Palatinado': 50,
    'Sarre': 54, 'Sajonia': 58, 'Sajonia-Anhalt': 62, 'Schleswig-Holstein': 66,
    'Turingia': 70
}

# 3. NORMALIZACIÓN DE NOMBRES DE PARTIDOS
# Pasamos la columna 0 a minúsculas para comparar sin errores de caja (Case-insensitive)
df_voto_raw[0] = df_voto_raw[0].str.strip().str.lower()

mapa_partidos_csv = {
    'cdu': 'CDU_CSU', 'csu': 'CDU_CSU',
    'spd': 'SPD',
    'grüne': 'VERDES', 'b90/gr': 'VERDES', 'bündnis 90/die grünen': 'VERDES',
    'fdp': 'FDP',
    'afd': 'AfD',
    'die linke': 'LINKE', 'die linke.': 'LINKE', 'linke': 'LINKE', 
    'pds': 'LINKE', 'linke liste/pds': 'LINKE'
}

# 4. EXTRACCIÓN
resultados = []

for nombre_csv_lower, p_final in mapa_partidos_csv.items():
    df_p = df_voto_raw[df_voto_raw[0] == nombre_csv_lower].copy()
    
    for estado, col_idx in col_map_voto.items():
        temp = df_p[[1, col_idx]].copy()
        temp.columns = ['Año', 'Voto']
        temp['Estado'] = estado
        temp['Partido'] = p_final
        
        # Limpieza de números (coma -> punto)
        temp['Voto'] = (temp['Voto'].str.replace('.', '', regex=False)
                                   .str.replace(',', '.', regex=False)
                                   .str.replace('–', '0').str.strip())
        
        temp['Voto'] = pd.to_numeric(temp['Voto'], errors='coerce').fillna(0)
        temp['Año'] = pd.to_numeric(temp['Año'], errors='coerce').fillna(0).astype(int)
        resultados.append(temp)

# 5. UNIFICACIÓN Y FORWARD FILL (Desde 1990)
df_largo = pd.concat(resultados)
df_largo = df_largo.groupby(['Estado', 'Año', 'Partido'])['Voto'].sum().reset_index()

final_list = []
partidos_finales = ['CDU_CSU', 'SPD', 'VERDES', 'FDP', 'LINKE', 'AfD']

for est in col_map_voto.keys():
    for part in partidos_finales:
        base = pd.DataFrame({'Estado': est, 'Año': range(1990, 2025), 'Partido': part})
        data = df_largo[(df_largo['Estado'] == est) & (df_largo['Partido'] == part)]
        merged = pd.merge(base, data[['Año', 'Voto']], on='Año', how='left')
        
        merged = merged.sort_values('Año')
        merged['Voto'] = merged['Voto'].ffill().fillna(0)
        final_list.append(merged)

df_final_votos = pd.concat(final_list)

# 6. PASAR A ANCHO (16 filas x N columnas de años/partidos)
df_wide = pd.DataFrame({'Estado': sorted(col_map_voto.keys())})
for part in partidos_finales:
    p_pivot = df_final_votos[df_final_votos['Partido'] == part].pivot(index='Estado', columns='Año', values='Voto')
    p_pivot.columns = [f"{c}_{part}" for c in p_pivot.columns]
    df_wide = pd.merge(df_wide, p_pivot, on='Estado')

# 7. VERIFICACIÓN
print("✅ Verificación de datos reales:")
# Comprobamos Sajonia (Sachsen) que es un bastión de Linke y ahora AfD
cols_ver = ['Estado', '1990_LINKE', '2013_AfD', '2021_LINKE', '2021_AfD']
print(df_wide.loc[df_wide['Estado'] == 'Sajonia', [c for c in cols_ver if c in df_wide.columns]])

✅ Verificación de datos reales:
     Estado  1990_LINKE  2013_AfD  2021_LINKE  2021_AfD
11  Sajonia         9.0       6.8         9.3      24.6


In [16]:
df_wide.head(16)

,Estado,1990_CDU_CSU,1991_CDU_CSU,1992_CDU_CSU,1993_CDU_CSU,1994_CDU_CSU,1995_CDU_CSU,1996_CDU_CSU,1997_CDU_CSU,1998_CDU_CSU,...,2015_AfD,2016_AfD,2017_AfD,2018_AfD,2019_AfD,2020_AfD,2021_AfD,2022_AfD,2023_AfD,2024_AfD
0,Baden-Wurtemberg,46.5,46.5,46.5,46.5,43.3,43.3,43.3,43.3,37.8,...,5.2,5.2,12.2,12.2,12.2,12.2,9.6,9.6,9.6,9.6
1,Baja Sajonia,44.3,44.3,44.3,44.3,41.3,41.3,41.3,41.3,34.1,...,3.7,3.7,9.1,9.1,9.1,9.1,7.4,7.4,7.4,7.4
2,Baviera,51.9,51.9,51.9,51.9,51.2,51.2,51.2,51.2,47.7,...,4.3,4.3,12.4,12.4,12.4,12.4,9.0,9.0,9.0,9.0
3,Berlin,39.4,39.4,39.4,39.4,31.4,31.4,31.4,31.4,23.7,...,4.9,4.9,12.0,12.0,12.0,12.0,9.4,9.4,9.4,9.4
4,Brandeburgo,36.3,36.3,36.3,36.3,28.1,28.1,28.1,28.1,20.8,...,6.0,6.0,20.2,20.2,20.2,20.2,18.1,18.1,18.1,18.1
5,Bremen,30.9,30.9,30.9,30.9,30.2,30.2,30.2,30.2,25.4,...,3.7,3.7,10.0,10.0,10.0,10.0,6.9,6.9,6.9,6.9
6,Hamburgo,36.6,36.6,36.6,36.6,34.9,34.9,34.9,34.9,30.0,...,4.2,4.2,7.8,7.8,7.8,7.8,5.0,5.0,5.0,5.0
7,Hesse,41.3,41.3,41.3,41.3,40.7,40.7,40.7,40.7,34.7,...,5.6,5.6,11.9,11.9,11.9,11.9,8.8,8.8,8.8,8.8
8,Mecklemburgo-Pomerania,41.2,41.2,41.2,41.2,38.5,38.5,38.5,38.5,29.3,...,5.6,5.6,18.6,18.6,18.6,18.6,18.0,18.0,18.0,18.0
9,Renania del Norte-Westfalia,40.5,40.5,40.5,40.5,38.0,38.0,38.0,38.0,33.8,...,3.9,3.9,9.4,9.4,9.4,9.4,7.3,7.3,7.3,7.3


In [17]:
# 1. Definición de las listas para la clasificación regional
estados_este = [
    'Brandeburgo', 'Mecklemburgo-Pomerania', 'Sajonia', 
    'Sajonia-Anhalt', 'Turingia'
]
# Berlín se suele tratar como categoría aparte por su estatus especial
# El resto se clasifican automáticamente como Oeste

# 2. Añadir el ID_Estado (del 1 al 16 basado en el orden alfabético de los estados)
# Primero ordenamos el dataframe por el nombre del Estado para que el ID sea consistente
df_wide = df_wide.sort_values('Estado').reset_index(drop=True)
df_wide.insert(0, 'ID_Estado', range(1, 17))

# 3. Crear la columna Region_Tipo
def clasificar_region(estado):
    if estado == 'Berlin':
        return 'Berlin'
    elif estado in estados_este:
        return 'Este'
    else:
        return 'Oeste'

df_wide.insert(2, 'Region_Tipo', df_wide['Estado'].apply(clasificar_region))

# 4. Verificación final del formato
print("✅ Dataset Final de Votos (df_wide) preparado.")
print(f"Dimensiones: {df_wide.shape} (16 filas, {df_wide.shape[1]} columnas)")
print("\nPrimeras filas con ID y Región:")
print(df_wide[['ID_Estado', 'Estado', 'Region_Tipo']].head())


✅ Dataset Final de Votos (df_wide) preparado.
Dimensiones: (16, 213) (16 filas, 213 columnas)

Primeras filas con ID y Región:
   ID_Estado            Estado Region_Tipo
0          1  Baden-Wurtemberg       Oeste
1          2      Baja Sajonia       Oeste
2          3           Baviera       Oeste
3          4            Berlin      Berlin
4          5       Brandeburgo        Este


In [18]:
# Aplicar un solo decimal a todas las columnas numéricas del DataFrame
df_wide = df_wide.round(1)

# Verificación rápida para asegurarnos de que el 7.99999 ahora es 8.0
print("✅ Columnas numéricas redondeadas a 1 decimal.")
# Mostramos una pequeña muestra de columnas de votos para confirmar
print(df_wide.iloc[:5, :10])

✅ Columnas numéricas redondeadas a 1 decimal.
   ID_Estado            Estado Region_Tipo  1990_CDU_CSU  1991_CDU_CSU  \
0          1  Baden-Wurtemberg       Oeste          46.5          46.5   
1          2      Baja Sajonia       Oeste          44.3          44.3   
2          3           Baviera       Oeste          51.9          51.9   
3          4            Berlin      Berlin          39.4          39.4   
4          5       Brandeburgo        Este          36.3          36.3   

   1992_CDU_CSU  1993_CDU_CSU  1994_CDU_CSU  1995_CDU_CSU  1996_CDU_CSU  
0          46.5          46.5          43.3          43.3          43.3  
1          44.3          44.3          41.3          41.3          41.3  
2          51.9          51.9          51.2          51.2          51.2  
3          39.4          39.4          31.4          31.4          31.4  
4          36.3          36.3          28.1          28.1          28.1  


In [19]:
def transformar_a_largo_seguro(df, nombre_valor):
    columnas_fijas = ['ID_Estado', 'Estado', 'Region_Tipo']
    
    # "Derretimos" el dataset
    df_long = df.melt(id_vars=columnas_fijas, var_name='Temp', value_name=nombre_valor)
    
    # 1. Extraemos los números usando r'' para evitar el SyntaxWarning
    df_long['Año_Extraido'] = df_long['Temp'].str.extract(r'(\d+)')
    
    # 2. Eliminamos las filas donde no se encontró un año (NaN) 
    df_long = df_long.dropna(subset=['Año_Extraido'])
    
    # 3. Convertimos a entero
    df_long['Año'] = df_long['Año_Extraido'].astype(int)
    
    # Limpiamos y ordenamos
    df_long = df_long.drop(columns=['Temp', 'Año_Extraido'])
    return df_long.sort_values(['ID_Estado', 'Año'])

# --- APLICACIÓN ---

df_pib_largo = transformar_a_largo_seguro(df_pib, 'PIB')
df_paro_largo = transformar_a_largo_seguro(df_paro_numero, 'Paro_Pct')
df_natalidad_largo = transformar_a_largo_seguro(df_natalidad, 'Natalidad')

# --- VOTOS ---
df_votos_long = df_wide.melt(id_vars=['ID_Estado', 'Estado', 'Region_Tipo'], 
                             var_name='Temp', value_name='Porcentaje')

# Usamos r'(\d+)' aquí también
df_votos_long['Año_Str'] = df_votos_long['Temp'].str.extract(r'(\d+)')
df_votos_long = df_votos_long.dropna(subset=['Año_Str'])
df_votos_long['Año'] = df_votos_long['Año_Str'].astype(int)

# Extraemos el partido
df_votos_long['Partido'] = df_votos_long['Temp'].str.split('_', n=1).str[1]

# Limpieza final de votos
df_votos_long = df_votos_long[['ID_Estado', 'Estado', 'Region_Tipo', 'Año', 'Partido', 'Porcentaje']]
df_votos_long = df_votos_long.sort_values(['ID_Estado', 'Año', 'Partido'])

# --- GUARDAR CSVs ---
config_guardado = {'index': False, 'sep': ';', 'decimal': ','}
df_pib_largo.to_csv('tabla_pib_largo.csv', **config_guardado)
df_paro_largo.to_csv('tabla_paro_largo.csv', **config_guardado)
df_natalidad_largo.to_csv('tabla_natalidad_largo.csv', **config_guardado)
df_votos_long.to_csv('tabla_votos_largo.csv', **config_guardado)

print("✅ Archivos guardados correctamente y sin errores de sintaxis.")

✅ Archivos guardados correctamente y sin errores de sintaxis.


In [20]:
# 1. Función rápida para limpiar y convertir a entero
def limpiar_a_entero(df, columna):
    # Convertimos a string, quitamos puntos de miles y cualquier espacio
    # Luego convertimos a float y finalmente a int para asegurar limpieza
    df[columna] = pd.to_numeric(
        df[columna].astype(str).str.replace('.', '', regex=False).str.strip(), 
        errors='coerce'
    ).fillna(0).astype(int)
    return df

# 2. Aplicamos la limpieza a los datasets correspondientes
df_natalidad_largo = limpiar_a_entero(df_natalidad_largo, 'Natalidad')

# También nos aseguramos de que el Año sea entero en todos (por si acaso)
for df in [df_pib_largo, df_paro_largo, df_natalidad_largo, df_votos_long]:
    df['Año'] = df['Año'].astype(int)

# 3. Exportar de nuevo con la codificación corregida (ñ) y sin decimales en natalidad
config_final = {
    'index': False, 
    'sep': ';', 
    'decimal': ',', 
    'encoding': 'utf-8-sig'
}

df_pib_largo.to_csv('tabla_pib_largo.csv', **config_final)
df_paro_largo.to_csv('tabla_paro_largo.csv', **config_final)
df_natalidad_largo.to_csv('tabla_natalidad_largo.csv', **config_final)
df_votos_long.to_csv('tabla_votos_largo.csv', **config_final)

print("✅ Natalidad convertida a número entero y archivos exportados con 'Año' correcto.")

✅ Natalidad convertida a número entero y archivos exportados con 'Año' correcto.


In [21]:
# 1. Limpieza radical del dataset de natalidad (problema con separador de miles)
# Vamos a tratar el dato como texto puro para evitar que Python mueva la coma

def limpieza_profunda_natalidad(df):
    # Convertimos a string y quitamos el .0 del final inmediatamente
    df['Natalidad'] = df['Natalidad'].astype(str).str.replace(r'\.0$', '', regex=True)
    
    # Si el número resultante tiene 7 cifras (como 1175280) y debería tener 6 (117528)
    # es probable que un punto decimal se haya ignorado.
    
    # Intentamos convertir a número limpio
    df['Natalidad'] = pd.to_numeric(df['Natalidad'], errors='coerce').fillna(0)
    
    # Regla de cordura: Si la natalidad de un estado es > 1.000.000, 
    # es un error de escala (ningún estado alemán tiene 1M de nacimientos al año)
    df.loc[df['Natalidad'] > 1000000, 'Natalidad'] = df['Natalidad'] / 10
    
    df['Natalidad'] = df['Natalidad'].astype(int)
    return df

# Aplicamos la corrección
df_natalidad_largo = limpieza_profunda_natalidad(df_natalidad_largo)

# 2. Verificación de control
check = df_natalidad_largo[(df_natalidad_largo['Estado'] == 'Baden-Wurtemberg') & (df_natalidad_largo['Año'] == 1991)]
print("📊 Nueva Verificación Baden-Wurtemberg 1991:")
print(check[['Estado', 'Año', 'Natalidad']])

# 3. Exportar con el formato de la Ñ corregido
config_final = {
    'index': False, 
    'sep': ';', 
    'decimal': ',', 
    'encoding': 'utf-8-sig'
}

df_natalidad_largo.to_csv('tabla_natalidad_largo.csv', **config_final)

📊 Nueva Verificación Baden-Wurtemberg 1991:
              Estado   Año  Natalidad
16  Baden-Wurtemberg  1991    1175280


In [22]:

# 1. Cargar los CSVs que acabamos de crear
# Usamos el separador y encoding que definimos antes
config = {'sep': ';', 'decimal': ',', 'encoding': 'utf-8-sig'}

df_pib = pd.read_csv('tabla_pib_largo.csv', **config)
df_paro = pd.read_csv('tabla_paro_largo.csv', **config)
df_nat = pd.read_csv('tabla_natalidad_largo.csv', **config)
df_votos = pd.read_csv('tabla_votos_largo.csv', **config)

# 2. Unir los indicadores básicos (PIB, Paro, Natalidad)
# Estos se unen fácil porque tienen 1 fila por Estado/Año
master_ml = pd.merge(df_pib, df_paro, on=['ID_Estado', 'Estado', 'Region_Tipo', 'Año'])
master_ml = pd.merge(master_ml, df_nat, on=['ID_Estado', 'Estado', 'Region_Tipo', 'Año'])

# 3. Preparar los VOTOS para ML
# En la tabla de votos, los partidos están hacia abajo (formato largo).
# Para ML los necesitamos hacia el lado (columnas).
votos_pivot = df_votos.pivot_table(
    index=['ID_Estado', 'Año'], 
    columns='Partido', 
    values='Porcentaje'
).reset_index()

# 4. Unión final: Indicadores + Votos
df_final_ml = pd.merge(master_ml, votos_pivot, on=['ID_Estado', 'Año'], how='left')

# 5. Ordenar por Estado y Año (importante para series temporales en ML)
df_final_ml = df_final_ml.sort_values(['ID_Estado', 'Año']).reset_index(drop=True)

# 6. Guardar el dataset listo para Scikit-Learn / PyTorch
df_final_ml.to_csv('Dataset_Final_ML_Alemania.csv', **config)

print("🚀 Dataset de Machine Learning listo.")
print(f"Dimensiones: {df_final_ml.shape}") # Debería ser algo como (480, nº_columnas)
print(df_final_ml.head())

🚀 Dataset de Machine Learning listo.
Dimensiones: (544, 13)
   ID_Estado            Estado Region_Tipo     PIB   Año  Paro_Pct  Natalidad  \
0          1  Baden-Wurtemberg       Oeste  243387  1991    159318    1175280   
1          1  Baden-Wurtemberg       Oeste  256691  1992    191970    1175590   
2          1  Baden-Wurtemberg       Oeste  254219  1993    281496    1179820   
3          1  Baden-Wurtemberg       Oeste  263619  1994    333416    1133980   
4          1  Baden-Wurtemberg       Oeste  272298  1995    328298    1124590   

   AfD  CDU_CSU   FDP  LINKE   SPD  VERDES  
0  0.0     46.5  12.3    0.3  29.1     5.7  
1  0.0     46.5  12.3    0.3  29.1     5.7  
2  0.0     46.5  12.3    0.3  29.1     5.7  
3  0.0     43.3   9.9    0.8  30.7     9.6  
4  0.0     43.3   9.9    0.8  30.7     9.6  


In [23]:
# Guardar el DataFrame df_paro_tasa en un archivo CSV
df_paro_tasa.to_csv('tabla_paro_tasa_final.csv', index=False, sep=';', encoding='utf-8-sig')

print("✅ Archivo 'tabla_paro_tasa_final.csv' guardado correctamente.")

✅ Archivo 'tabla_paro_tasa_final.csv' guardado correctamente.


In [24]:
# 1. Eliminar la columna de 2025 (si existe en los nombres con el sufijo)
col_2025 = "2025_tasa_paro"
if col_2025 in df_paro_tasa.columns:
    df_paro_tasa = df_paro_tasa.drop(columns=[col_2025])

# 2. Transformar a formato largo (Melt)
# Mantenemos fijas las columnas de identificación
df_paro_largo = df_paro_tasa.melt(
    id_vars=['ID_Estado', 'Estado', 'Region_Tipo'], 
    var_name='Año', 
    value_name='Tasa_Paro'
)

# 3. Limpiar la columna 'Año' para que sea solo el número (quitar el "_tasa_paro")
df_paro_largo['Año'] = df_paro_largo['Año'].str.replace('_tasa_paro', '').astype(int)

# 4. Ordenar para que sea fácil de leer (Estado y luego Año)
df_paro_largo = df_paro_largo.sort_values(by=['ID_Estado', 'Año']).reset_index(drop=True)

# 5. Guardar el nuevo archivo
df_paro_largo.to_csv('tabla_paro_tasa_largo.csv', index=False, sep=';', encoding='utf-8-sig')

print("✅ Transformación a formato largo completada (sin 2025).")
print(df_paro_largo.head())

✅ Transformación a formato largo completada (sin 2025).
   ID_Estado            Estado Region_Tipo   Año  Tasa_Paro
0          1  Baden-Wurtemberg       Oeste  1991        3.7
1          1  Baden-Wurtemberg       Oeste  1992        4.4
2          1  Baden-Wurtemberg       Oeste  1993        6.3
3          1  Baden-Wurtemberg       Oeste  1994        7.5
4          1  Baden-Wurtemberg       Oeste  1995        7.4


In [25]:
# Guardar usando coma como separador decimal para que Power BI en español lo reconozca
df_paro_largo.to_csv('tabla_paro_tasa_largo.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')